# Phase 9.2c: final development fit
Run cells in order. The notebook selects settings on the frozen fourth inner fold, then fits the two AI candidates using eligible development labels only. It creates development-only artifacts and does not assess an untouched test.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
from io import BytesIO
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    REPO_ROOT = Path('/content/HalalifyAI')
    if not (REPO_ROOT / 'AGENTS.md').is_file():
        subprocess.run(['git', 'clone', '--branch', 'feature/model-evaluation', '--single-branch', 'https://github.com/frager098/HalalifyAI.git', str(REPO_ROOT)], check=True)
    code_paths = [Path('configs/phase9_final_fit_v1.json'), Path('src/model_training/phase9_final_fit.py')]
    missing_code = [p for p in code_paths if not (REPO_ROOT / p).is_file()]
    if missing_code:
        from google.colab import files
        print('Upload phase9_final_fit_code.zip containing:', *map(str, missing_code))
        archive = files.upload().get('phase9_final_fit_code.zip')
        if archive is None:
            raise FileNotFoundError('Upload phase9_final_fit_code.zip')
        with zipfile.ZipFile(BytesIO(archive)) as bundle:
            expected = {p.as_posix() for p in missing_code}
            if set(bundle.namelist()) != expected or any('..' in Path(n).parts or Path(n).is_absolute() for n in bundle.namelist()):
                raise ValueError(f'Code ZIP must contain exactly: {sorted(expected)}')
            bundle.extractall(REPO_ROOT)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'scipy', 'scikit-learn', 'joblib'], check=True)
else:
    start = Path.cwd().resolve()
    REPO_ROOT = next((p for p in [start, *start.parents] if (p / 'AGENTS.md').is_file() and (p / 'src').is_dir() and (p / 'configs').is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Open Jupyter inside the HalalifyAI repository.')
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Environment:', 'Colab' if IN_COLAB else 'Local', 'Repository:', REPO_ROOT)


In [ ]:
import json
from src.model_training.phase9_final_fit import verify_inputs
CONFIG = Path('configs/phase9_final_fit_v1.json')
spec = json.loads(CONFIG.read_text(encoding='utf-8'))
required = [Path(item['path']) for item in spec['inputs'].values()]
print('Required inputs:')
for path in required:
    print(path, 'present' if path.is_file() else 'MISSING')
if IN_COLAB and any(not path.is_file() for path in required):
    from google.colab import files
    missing = {p.as_posix() for p in required if not p.is_file()}
    print('Upload phase9_final_fit_inputs.zip containing exactly these missing paths:', *sorted(missing), sep='\n')
    archive = files.upload().get('phase9_final_fit_inputs.zip')
    if archive is None:
        raise FileNotFoundError('Upload phase9_final_fit_inputs.zip')
    with zipfile.ZipFile(BytesIO(archive)) as bundle:
        if set(bundle.namelist()) != missing or any('..' in Path(n).parts or Path(n).is_absolute() for n in bundle.namelist()):
            raise ValueError(f'Input ZIP must contain exactly: {sorted(missing)}')
        bundle.extractall(REPO_ROOT)
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Restore inputs at: ' + ', '.join(missing))
OUTPUT = Path(spec['output'])
if OUTPUT.exists():
    raise FileExistsError(f'Preserve existing results: {OUTPUT}')
VERIFIED = verify_inputs(CONFIG)
print('Preflight passed:', len(VERIFIED[1]), 'dataset rows; selected validation fold:', spec['selection']['validation_fold'])


In [ ]:
from src.model_training.phase9_final_fit import run_final_fit
manifest = run_final_fit(CONFIG, OUTPUT, VERIFIED)
print('Development-only fit complete. Files:', *sorted(str(p) for p in OUTPUT.iterdir()), sep='\n')
for task in ('return', 'risk'):
    print(task, 'selected:', manifest['selection'][task]['selected_parameters'], 'training rows:', manifest['final_fit'][task]['training']['rows'])
print('No untouched-test result was produced.')


In [ ]:
import shutil
print(json.dumps({task: manifest['artifacts'][task]['sha256'] for task in ('return', 'risk')}, indent=2))
if IN_COLAB:
    from google.colab import files
    archive_path = Path(str(OUTPUT) + '.zip')
    if archive_path.exists():
        raise FileExistsError(f'Preserve existing download ZIP: {archive_path}')
    archive = shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT.parent, base_dir=OUTPUT.name)
    files.download(archive)
